In [43]:
import numpy as np 

In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import tensorflow as tf
import cv2


In [45]:
data_set=pd.read_csv("churn_modelling.csv")

In [46]:
X=data_set.iloc[:,3:13]
Y=data_set.iloc[:,13]

In [47]:
# creating dummy data 
geography=pd.get_dummies(X["Geography"], drop_first=True)
gender=pd.get_dummies(X["Gender"], drop_first=True)

In [48]:
# concatinating the frame
X=pd.concat([X, geography, gender], axis=1)

In [49]:
X=X.drop(["Geography","Gender"],axis=1)

In [50]:
from sklearn.model_selection import train_test_split

In [51]:
X_train, X_test, Y_train, Y_test=train_test_split(X,Y, test_size=0.2, random_state=0)

In [52]:
# feature scalling
from sklearn.preprocessing import StandardScaler

sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.fit_transform(X_test)

In [25]:
# perform hyperparameter tunning
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

In [55]:
# part 2: Now let's make ANN
import keras
from keras.models import Sequential
from keras.layers import Dense, Activation, Embedding, Flatten, BatchNormalization
from keras.layers import LeakyReLU, PReLU, ELU
from keras.activations import sigmoid
from keras.layers import Dropout

In [60]:
# Funtion to initalizing the ANN this is general code 
def create_model(layers, activation):
    model=Sequential()
    for i, nodes in enumerate(layers):
        if i==0:
            model.add(Dense(nodes, input_dim=X_train.shape[1]))
            model.add(Activation(activation))
            model.add(Dropout(0.3))
        else:
            model.add(Dense(nodes))
            model.add(Activation(activation))
            model.add(Dropout(0.3))

    model.add(Dense(units=1, kernel_initializer='glorot_uniform', activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model=KerasClassifier(model=create_model, verbose=0)
        

In [61]:
# Hyper parameter
layers=[[20],[40,20],[45,30,15]]
activation=['sigmoid','relu']
param_grid=dict(model__layers=layers, model__activation=activation, batch_size=[128,256], epochs=[30])
grid=GridSearchCV(estimator=model, param_grid=param_grid, cv=5)

grid_result=grid.fit(X_train,Y_train)


D:\Data-Science-Projects\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
D:\Data-Science-Projects\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
D:\Data-Science-Projects\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regula

D:\Data-Science-Projects\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


D:\Data-Science-Projects\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
D:\Data-Science-Projects\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
D:\Data-Science-Projects\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regula

In [64]:
print(grid_result.best_score_, grid_result.best_params_)

0.8550000000000001 {'batch_size': 128, 'epochs': 30, 'model__activation': 'relu', 'model__layers': [45, 30, 15]}


In [65]:
pred_y=grid.predict(X_test)
Y_pred=(pred_y>0.5).astype(int)

In [66]:
from sklearn.metrics import confusion_matrix, accuracy_score 

In [68]:
cm=confusion_matrix(Y_pred, Y_test)
cm

array([[1540,  214],
       [  55,  191]])

In [70]:
score=accuracy_score(Y_pred, Y_test)
score

0.8655